# U7-01 · AutoML con FLAML, registro de experimentos y API con FastAPI

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 7*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894

In [ ]:
%pip install -q "flaml[automl]" fastapi httpx
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["bicis.py", "hour.csv"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://especialidades-formativas.github.io/ifcd107/recursos/datos/u7/" + f, f)

## 7.1 · Qué automatiza el AutoML

`v1_automl.py`

In [ ]:
import numpy as np
from flaml import AutoML
from sklearn.metrics import mean_absolute_error
from bicis import X, y, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
auto = AutoML()
auto.fit(X.loc[ent, cols], y[ent], task="regression", metric="mae",
         time_budget=120,                 # segundos de búsqueda
         eval_method="cv", n_splits=4, split_type="time",
         estimator_list=["lgbm", "rf", "extra_tree", "xgboost"],
         seed=0, verbose=0)
print("mejor algoritmo:", auto.best_estimator)
print("EAM de validación:", round(auto.best_loss, 1))
for k, v in auto.best_config.items():
    print(f"  {k}: {round(v, 4) if isinstance(v, float) else v}")
p = auto.predict(X.loc[pru, cols])
print(f"EAM de prueba: {mean_absolute_error(y[pru], p):.1f}")

## 7.2 · Del cuaderno a producción: MLOps

`v2_registro.py`

In [ ]:
import json
import time
import hashlib
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from bicis import X, y, h, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
val = ent & (h.dteday >= "2012-04-01").to_numpy()  # validación
aju = ent & ~val
registro = []
for tasa in [0.03, 0.05, 0.1]:            # cada experimento, anotado
    params = {"loss": "poisson", "max_iter": 500,
              "learning_rate": tasa, "random_state": 0}
    m = HistGradientBoostingRegressor(**params)
    m.fit(X.loc[aju, cols], y[aju])
    eam = mean_absolute_error(y[val], m.predict(X.loc[val, cols]))
    archivo = f"modelo_{tasa}.joblib"
    joblib.dump(m, archivo)
    huella = hashlib.sha256(open(archivo, "rb").read()).hexdigest()
    registro.append({"fecha": time.strftime("%Y-%m-%d"),
                     "datos": "hour.csv hasta 2012-03-31",
                     "variables": cols, "parametros": params,
                     "eam_val": round(eam, 1),
                     "archivo": archivo, "sha256": huella[:12]})
json.dump(registro, open("registro.json", "w"), indent=1)
for r in registro:
    print(r["parametros"]["learning_rate"], r["eam_val"],
          r["archivo"], r["sha256"])
mejor = min(registro, key=lambda r: r["eam_val"])
m = joblib.load(mejor["archivo"])        # reentrenar con todo
m.fit(X.loc[ent, cols], y[ent]); joblib.dump(m, "produccion.joblib")
e = mean_absolute_error(y[pru], m.predict(X.loc[pru, cols]))
print(f"elegido {mejor['archivo']}; EAM de prueba {e:.1f}")

`v3_servicio.py`

In [ ]:
import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

modelo = joblib.load("produccion.joblib")
app = FastAPI(title="Previsión de alquileres de bicicletas")

class Hora(BaseModel):                    # qué datos se aceptan
    hora: int = Field(ge=0, le=23)
    laborable: int = Field(ge=0, le=1)
    temp_c: float
    humedad: float = Field(ge=0, le=100)
    viento_kmh: float = Field(ge=0)
    tiempo: int = Field(ge=1, le=3)
    mes: int = Field(ge=1, le=12)
    año: int = Field(ge=0, le=1)
    dia_semana: int = Field(ge=0, le=6)
    festivo: int = Field(ge=0, le=1)

@app.post("/prevision")
def prevision(h: Hora):
    x = pd.DataFrame([h.model_dump()])
    return {"alquileres": round(float(modelo.predict(x)[0]), 1)}

cliente = TestClient(app)        # probar sin arrancar servidor
datos = dict(hora=8, laborable=1, temp_c=20, humedad=60,
             viento_kmh=10, tiempo=1, mes=9, año=1, dia_semana=2,
             festivo=0)
r = cliente.post("/prevision", json=datos)
print(r.status_code, r.json())
r = cliente.post("/prevision", json=dict(datos, hora=25))
print(r.status_code, r.json()["detail"][0]["msg"])

## 7.3 · Vigilar el modelo

`v4_vigilar.py`

In [ ]:
import joblib
import pandas as pd
from scipy.stats import ks_2samp
from bicis import X, y, h, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
m = joblib.load("produccion.joblib")
p = m.predict(X.loc[pru, cols])
d = pd.DataFrame({"mes": h.dteday[pru].dt.to_period("M"),
                  "error": (y[pru] - p).abs(), "real": y[pru]})
umbral = 1.25 * 37.5            # 25 % peor que en validación
for mes, g in d.groupby("mes"):
    e = g.error.mean()
    aviso = "  <- revisar" if e > umbral else ""
    print(f"{mes}: EAM {e:5.1f},"
          f" media real {g.real.mean():5.1f}{aviso}")
for v in ["temp_c", "humedad"]:           # ¿han cambiado los datos?
    r = ks_2samp(X.loc[ent, v], X.loc[pru, v])
    print(f"{v}: estadístico KS {r.statistic:.3f},"
          f" p = {r.pvalue:.1e}")

**Prueba.** En `v1_automl.py`, añade `"histgb"` a la lista de algoritmos o aumenta el presupuesto a 300 segundos. ¿Mejora el EAM de prueba?